# Langchain Middleware And Runtime State
<img src="../../assets/runtime_params.png" width="1000" height="400">
<img src="../../assets/agentic_components.png" width="1000" height="400">


In [1]:
import os
import json
from dotenv import load_dotenv
from typing import Literal
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langgraph.store.memory import InMemoryStore
from langchain.agents.middleware import dynamic_prompt


load_dotenv()

True

In [2]:
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY environment variable is not set.")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

model = ChatOpenAI(model="gpt-5-nano")

## Dynamic Prompting
* Dynamically modify the system prompt at runtime to inject context, user-specific instructions, or other information before each model call.
* Use the `system_message` field on `ModelRequest` to read and modify the system prompt.
* It contains a `SystemMessage` object.

In [13]:
class UserContext(BaseModel):
    user_name: str = Field(..., description="The unique identifier for the user.")
    user_role: Literal["developer", "manager"] = Field(..., description="The role of the user, which can be either 'developer' or 'manager'.")

In [22]:
@dynamic_prompt
def personalize_the_prompt(request):
    user_role = request.runtime.context.user_role
    system_prompt=''
    if user_role == "developer":
        system_prompt = """
        You are an expert developer.

        Give technical answers with code examples.
        Assume familiarity with programming concepts.
        """
    if user_role == "manager":
        system_prompt = """
        You are assisting an engineering manager.

        Focus on architecture, risks, trade-offs,
        timelines, and business impact.
        Avoid unnecessary implementation details.
        """
    print("\n========== SYSTEM PROMPT ==========")
    print(system_prompt)
    print("===================================\n")
    return system_prompt

In [23]:
dynamic_system_prompt_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[],
    middleware=[personalize_the_prompt],
    context_schema=UserContext,
)

In [24]:
result = dynamic_system_prompt_agent.invoke({"messages": [HumanMessage(content="Write a Python function to print hello world")]},
                                            context={"user_name": "Shivam", "user_role": "developer"})


========== SYSTEM PROMPT ==========

        You are an expert developer.

        Give technical answers with code examples.
        Assume familiarity with programming concepts.
        



In [25]:
result = dynamic_system_prompt_agent.invoke({"messages": [HumanMessage(content="Write a sentence about the importance of code reviews")]},
                                            context={"user_name": "Satyam", "user_role": "manager"})


========== SYSTEM PROMPT ==========

        You are assisting an engineering manager.

        Focus on architecture, risks, trade-offs,
        timelines, and business impact.
        Avoid unnecessary implementation details.
        

